In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.optim as optim
from utils.SplitData import split_data_Train_Val_Test
from torch.utils.tensorboard import SummaryWriter # type: ignore

from model.trace import TRACE
from dataset.otto_final import TraceOttoDataset
from utils.EarlyStopping import EarlyStopping
from utils.feature_engineering import get_between_features, get_elapsed_feature
from sklearn.metrics import f1_score,precision_score,recall_score
from utils.training_utils import search_best_f1_thr, update_binary_metrics

import os
import numpy as np
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
#DataSet    
dataset_processed = TraceOttoDataset(
        file_name='train.jsonl',
        input_seq_len=64,
        min_timestamps_per_sample=32,max_samples=1000000)

print(dataset_processed.__getitem__(2))


In [ ]:
def see_target_temporal_windows(dataset):
    target_lengths = []
    temporal_gaps = []

    for session in dataset.session:
        input_part, target_part = dataset.__split_input_target__(session)

        target_lengths.append(len(target_part["timestamps"]))
        gap = target_part["timestamps"][0] - input_part["timestamps"][-1]
        temporal_gaps.append(gap)

    print("Target length:")
    print("min of timestamps in the target:", min(target_lengths))
    print("mean: of timestamps in the target", sum(target_lengths) / len(target_lengths))
    print("max: of timestamps in the target", max(target_lengths))

    print("Temporal gap in miliseconds:")
    print("min:", min(temporal_gaps))
    print("mean:", sum(temporal_gaps) / len(temporal_gaps))
    print("max:", max(temporal_gaps))


In [ ]:
def inspect_purchase_temporal_windows(dataset):
    purchase_counts = []
    purchase_gaps = []

    for session in dataset.session:
        input_part, target_part = dataset.__split_input_target__(session)

        target_orders = np.asarray(target_part["type"])
        input_ts = np.asarray(input_part["timestamps"])
        target_ts = np.asarray(target_part["timestamps"]) 
        purchase_ts = target_ts[(target_orders == 3)]

        if purchase_ts.size == 0:
            continue

        purchase_counts.append(int(purchase_ts.size))
        purchase_gaps.append(int(purchase_ts[0] - input_ts[-1]))

    if not purchase_counts:
        print("No purchases found in any target window.")
        return

    print("Purchase count per target:")
    print("min:", min(purchase_counts))
    print("mean:", np.mean(purchase_counts))
    print("max:", max(purchase_counts))

    print("\nPurchase temporal gap:")
    print("min:", min(purchase_gaps))
    print("mean:", np.mean((purchase_gaps)))
    print("max:", max(purchase_gaps))


In [ ]:
see_target_temporal_windows(dataset_processed)

In [ ]:
see_target_temporal_windows(dataset_processed)

In [ ]:
inspect_purchase_temporal_windows(dataset_processed)

In [ ]:
task_train = "PD1"

if task_train not in {"ATC", "SAT", "PD1", "RA1"}:
    raise ValueError(f"Invalid task '{task_train}'. Choose ATC, SAT, PD1 or RA1.")

In [ ]:
#Split the Data into Training_loader, Validation_loader and test_loaders
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=128)
    
#calling the max aid and type for combating the Out of Range Error -> Learning Embeddings
max_aid = max(
    session[0]["aid"].max().item()
    for session in dataset_processed
)
max_type = max(
    session[0]["type"].max().item()
    for session in dataset_processed
)
num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
    
#Initialize the Model TRACE(Model Architecture TRACE paper Part 2.3)
trace_model = TRACE(
        num_embeddings_aid=num_embeddings_aid,
        num_embeddings_event_type=num_embeddings_event_type,
        embedding_dim=32,
        num_classes=1
    )

In [ ]:

trace_model = trace_model.to(device)
optimizer = optim.AdamW(trace_model.parameters(), lr=1e-4, weight_decay=1e-4)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,
                                                        cooldown=1,
                                                        mode="max",
                                                        factor=0.5,
                                                        patience=2,
                                                        min_lr=1e-6)
early_stopping = EarlyStopping(patience=7,
                                min_delta=1e-4,
                                mode="max",
                                path=f"Model_TRACE_checkpoint_{task_train}_task.pt")
    
#Summary Writer for tensorBoard
tensor_board_writer = SummaryWriter(log_dir=f"runs/{task_train}/")
    

In [ ]:
"""
Shape of train_loader, val_loader, Test_loader
"""
for inputs_train, targets_train in train_loader:
    print("The version of train_loader")
    print(inputs_train.keys())
    print(targets_train.keys())
    print(
        f"Shape Aids: {inputs_train['aid'].shape}, "
        f"Shape Timestamps: {inputs_train['timestamps'].shape}, "
        f"Shape Type: {inputs_train['type'].shape}"
    )
    break  

for inputs_val, targets_val in validation_loader:
    
    print("The version of val_loader")
    print(inputs_val.keys())
    print(targets_val.keys())
    
    print(
        f"Shape Aids: {inputs_val['aid'].shape}, "
        f"Shape Timestamps: {inputs_val['timestamps'].shape}, "
        f"Shape Type: {inputs_val['type'].shape}"
    )
    break
for inputs_test, targets_test in test_loader:
    print("The version of train_loader")
    print(inputs_test.keys())
    print(targets_test.keys())
    print(
        f"Shape Aids: {inputs_test['aid'].shape}, "
        f"Shape Timestamps: {inputs_test['timestamps'].shape}, "
        f"Shape Type: {inputs_test['type'].shape}"
    )
    break

print("==================================SIZE=================================== ")
print(f"Train LOADER {len(train_loader)}")  
print(f"Validation LOADER {len(validation_loader)}")  
print(f"Test LOADER {len(test_loader)}")  


In [ ]:
print("Started the Training")
    #Figthing Data Imbalanced
labels_list = []
for inputs, targets in train_loader:
    assert task_train in targets, f"Unknown task: {task_train}"
    labels_list.append(targets[task_train].view(-1)) #(Batch, )
        
labels = torch.cat(labels_list, dim=0) #(N, )           
    
#Number of positives in the train_loader
num_pos = (labels == 1).sum().item()
    
#Number of Negatives in the train_loader
num_neg = (labels == 0).sum().item()
    
ratio = num_neg / max(num_pos, 1)
ratio = min(ratio, 30.0)
    
#smoothed_weight = torch.tensor([ratio], device=device)
    
print("Train pos/neg:", num_pos, num_neg)
    
##adding for smoothing the weights only for Training 
#criterion_train = torch.nn.BCEWithLogitsLoss(pos_weight=smoothed_weight) 
    
criterion_validation = torch.nn.BCEWithLogitsLoss()
    
w_pos = torch.tensor([ratio], device=device).float() 
w_neg = torch.tensor([1.0], device=device).float()

print("w_pos and w_neg", w_pos, w_neg)
#Learning Rate Scheduler
#To Save the Best F1 for the Model
best_val_f1 = -1.0
best_global_thr = 0.5
    
    
num_epochs = 40
for epoch in range(num_epochs):
        
    #F1 Score training
    all_train_y_true = []
    all_train_y_pred = []
    all_val_probs = []
    all_val_y_true = []
            
    # -------------------------------TRAINING ---------------------------
    #Initializing the training variables
    trace_model.train()
    epoch_loss = 0.0
    correct_train = 0
    total_train = 0
            
    for inputs_train, targets_train in train_loader:
                
        target_train = targets_train[task_train].unsqueeze(1).to(device).float()
        #Changing the Inputs -> to have GPU for JupyterGPUHub
        inputs_train = {k: v.to(device) for k, v in inputs_train.items()}
        #Calculation of the timestamps(Feature Engineer Trace Paper Part 2.2)
        delta_elapsed = get_elapsed_feature(inputs_train["timestamps"]).to(device).float()
        delta_between = get_between_features(inputs_train["timestamps"]).to(device).float()
                
        optimizer.zero_grad(set_to_none=True)
        #Predictions of the model
        logits_train = trace_model(
                inputs_train["aid"],
                inputs_train["type"],
                delta_elapsed,
                delta_between
            )
            
        weights = torch.where(target_train == 1.0, w_pos, w_neg)

        #Calculation loss for Training using BCEWithLogitsLoss
        loss_training = F.binary_cross_entropy_with_logits(logits_train,target_train, weight=weights)
        loss_training.backward()
        optimizer.step()
                
        epoch_loss += loss_training.item()
                
        # ============(Prediction, calculation of Accuracy) ============
        correct_train, total_train = update_binary_metrics(logits_train,target_train,correct_train,total_train,all_train_y_true,all_train_y_pred)
            
            
    #Training Loss and Accuracy 
    train_loss = epoch_loss / len(train_loader)
    train_acc = correct_train / max(total_train, 1)
            
    #F1 Score for training
    all_train_y_true = torch.cat(all_train_y_true).numpy().ravel()
    all_train_y_pred = torch.cat(all_train_y_pred).numpy().ravel()
    train_f1 = f1_score(all_train_y_true, all_train_y_pred, zero_division=0)
            
    #TensorBoard Writing
    tensor_board_writer.add_scalar("Train/F1", train_f1, epoch)
    tensor_board_writer.add_scalar("Training/Loss", train_loss, epoch)
    tensor_board_writer.add_scalar("Train/Acc", train_acc, epoch)
            
    
    
    # -------------------------------VALIDATION---------------------------
    #Initializing the validation variables    
    trace_model.eval()
    val_loss = 0.0
            
        
        
    with torch.no_grad():
        for inputs_val, targets_val in validation_loader:
            target_val = targets_val[task_train].unsqueeze(1).to(device).float()
    
            inputs_val = {k: v.to(device) for k, v in inputs_val.items()}
    
            delta_elapsed = get_elapsed_feature(inputs_val["timestamps"]).to(device).float()
            delta_between = get_between_features(inputs_val["timestamps"]).to(device).float()
    
            logits_val = trace_model(
                inputs_val["aid"],
                inputs_val["type"],
                delta_elapsed,
                delta_between
            )
                
            loss_validation = criterion_validation(logits_val, target_val)
            val_loss += loss_validation.item()

            #Logits converted to sigmoid
            append_probs_and_true(logits_val, target_val, all_val_probs, all_val_y_true)
    
    # ----Concatonate the Probabilities and true labels ----
    all_val_y_true = torch.cat(all_val_y_true).numpy().ravel()
    all_val_probs = torch.cat(all_val_probs).numpy().ravel()
    
    #Generate 99 possible threshold values from 0.1 to 0.99 (steps of 0.01).
    thresholds = np.linspace(0.01, 0.99, 99)
       
    best_f1, best_thr = search_best_f1_thr(all_val_probs, all_val_y_true, thresholds)
        
    #Looking for the Best F1 Score and threshold
    val_f1 = best_f1
    threshold = best_thr
        
    # Generate final predictions using the newly discovered optimal threshold
    val_pred = (all_val_probs >= threshold).astype(int)
        
    # Calculate additional metrics Precision and Recall at this specific optimal threshold
    val_precision = precision_score(all_val_y_true, val_pred, zero_division=0)
    val_recall = recall_score(all_val_y_true, val_pred, zero_division=0)
        
    print(
            f"Thr={threshold:.3f} | "
            f"P={val_precision:.3f} | "
            f"R={val_recall:.3f} | "
            f"F1={val_f1:.3f}"
    )

        
    # If the F1 score of this epoch is the best seen so far across all epochs,
    # we update the global "Best Model" variables to ensure we save the right threshold.
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_global_thr = threshold
    
        
    #Validation Loss
    val_loss /= len(validation_loader)
        
    #calculates the optimized Accuracy based on the best threshold found
    val_acc_best_thr = ((all_val_probs >= threshold).astype(int) == all_val_y_true.astype(int)).mean()
        
    #TensorBoard
    tensor_board_writer.add_scalar("Val/Loss", val_loss, epoch)
    tensor_board_writer.add_scalar("Val/Acc_best_thr", val_acc_best_thr, epoch)
    tensor_board_writer.add_scalar("Val/F1", val_f1, epoch)
    tensor_board_writer.add_scalar("Val/Best_Threshold", threshold, epoch)
    tensor_board_writer.add_scalar("Val/Best_Global_Threshold", best_global_thr, epoch)
    tensor_board_writer.add_scalar("Val/Precision", val_precision, epoch)
    tensor_board_writer.add_scalar("Val/Recall", val_recall, epoch)

    lr_scheduler.step(val_f1)
                            
    print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | "
            f"BestThr: {threshold:.3f} | Val Acc best threshold: {val_acc_best_thr:.4f} "
        )

    #Print the Current Learning rate after the Lr
    current_lr = optimizer.param_groups[0]["lr"]
    tensor_board_writer.add_scalar("LR", current_lr, epoch)
    print(f"This is the LR: {current_lr}")
            
    #Early Stopping
    early_stopping(val_f1.__float__(), trace_model)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break
        
tensor_board_writer.close()

In [ ]:
early_stopping.load_best_weights(trace_model)
    
#Save the total Model after the training
torch.save({
    "model_state_dict": trace_model.state_dict(),
    "best_val_f1": best_val_f1,
    "best_global_threshold": best_global_thr,
}, f"Model_TRACE_{task_train}_FinalVersion_SingleTask.pt")


